# Phase 6: Explanations

**Goal:** for every flagged account, write a short plain-English reason, built from templates so it can never invent facts.

**How it works (3 ideas):**
1. Every feature is already scaled (Phase 4), so a scaled value means "how many standard deviations from a typical account".
2. For each account, find the features that are most unusual **in the suspicious direction**, up to two, and only if they are really unusual.
3. Fill a sentence template for each one, using the account's **real (unscaled) values**.

**Rule to remember:** *rank* with the scaled values, *write* with the real values.

Labels are **not** used until Step 8 (the quality check). Restart the kernel and run top to bottom.

## Setup: load the data with the same code the API will use (Phase 4 functions)

In [63]:
import sys
import joblib
import numpy as np
import pandas as pd

sys.path.append("..")
from src.features import build_features, prepare_for_model, FEATURES
from src.models import zscore_scores, isolation_forest_scores, flag_top

scaler = joblib.load("../artifacts/scaler.pkl")
model = joblib.load("../artifacts/isolation_forest.pkl")

raw = pd.read_csv("../data/features.csv")

feats = build_features(raw)                  # real values, one row per account (for the sentences)
scaled = prepare_for_model(feats, scaler)    # scaled values (for ranking)

feature_cols = FEATURES
print(feats.shape, scaled.shape)
feats.head(3)

(3099, 5) (3099, 5)


,account_age_days,trades_per_day,spike_ratio,linked_account_count,timing_regularity
account_id,,,,,
ACC00000,213,9.15,0.706019,0,0.874
ACC00001,394,5.27,1.203196,1,0.571
ACC00002,41,8.72,0.794895,0,1.322


In [64]:
# Check: same numbers as the saved Phase 4 feature matrix (should print True)
matrix = pd.read_csv("../data/feature_matrix.csv", index_col="account_id")
print(np.allclose(scaled.values, matrix[feature_cols].values))

True


## Step 1: Sentence templates

One short sentence piece per feature. The function also fixes grammar ("1 day" not "1 days") and avoids numbers that mean nothing to a reviewer (timing is described in words).

In [65]:
def make_piece(col, value):
    if col == "account_age_days":
        days = int(round(value))
        if days <= 0:
            return "it was created today"
        word = "day" if days == 1 else "days"
        return f"it is only {days} {word} old"
    if col == "trades_per_day":
        return f"it makes {value:.0f} trades per day"
    if col == "spike_ratio":
        return f"its trading is {value:.1f}x its usual pace"
    if col == "linked_account_count":
        count = int(round(value))
        word = "account" if count == 1 else "accounts"
        return f"it shares a device or IP with {count} other {word}"
    if col == "timing_regularity":
        return "its trade timing is unusually regular"

print(make_piece("account_age_days", 1))
print(make_piece("linked_account_count", 9))

it is only 1 day old
it shares a device or IP with 9 other accounts


## Step 2: How unusual is each feature?

Only one direction is suspicious for each feature:

| Feature | Suspicious when |
|---|---|
| account_age_days | low |
| trades_per_day | high |
| spike_ratio | high |
| linked_account_count | high |
| timing_regularity | low |

We store this as `+1` (high is suspicious) or `-1` (low is suspicious). Multiplying a scaled value by it makes **bigger always mean more suspicious**.

In [66]:
direction = {
    "account_age_days":     -1,
    "trades_per_day":       +1,
    "spike_ratio":          +1,
    "linked_account_count": +1,
    "timing_regularity":    -1,
}

def suspicion(scaled_row):
    """How suspicious each feature is for one account (bigger = more suspicious)."""
    return {col: scaled_row[col] * direction[col] for col in feature_cols}

# Example: one trial-abuse style account
example = "ACC00526"
print(pd.Series(suspicion(scaled.loc[example])).round(2))
print()
print(feats.loc[example])

account_age_days         1.70
trades_per_day           5.97
spike_ratio             12.58
linked_account_count     0.77
timing_regularity        0.03
dtype: float64

account_age_days          1.000000
trades_per_day          879.690000
spike_ratio              27.997772
linked_account_count      1.000000
timing_regularity         0.980000
Name: ACC00526, dtype: float64


## Steps 3 to 5: Pick the top two, build the sentence, handle edge cases

- Take the two most suspicious features.
- Keep a feature only if its score is at least `MIN_SCORE` (1 standard deviation). Otherwise a normal-looking feature could appear in the explanation and mislead the reviewer.
- If nothing is unusual enough, return an honest fallback sentence.
- A zero baseline is already handled in `build_features()` (spike ratio becomes 1.0, "normal pace"), so there is no "infinite pace" here.

In [67]:
MIN_SCORE = 1.0

def explain_account(feat_row, scaled_row):
    score = suspicion(scaled_row)

    top_two = sorted(score, key=score.get, reverse=True)[:2]
    top_two = [col for col in top_two if score[col] >= MIN_SCORE]

    if len(top_two) == 0:
        return "No feature looks clearly suspicious on its own; review the account's raw values."

    pieces = [make_piece(col, feat_row[col]) for col in top_two]
    sentence = " and ".join(pieces)
    return sentence[0].upper() + sentence[1:] + "."

In [68]:
# Test 1: an obvious abuser
print(explain_account(feats.loc["ACC00526"], scaled.loc["ACC00526"]))

# Test 2: zero baseline, sent through the real pipeline (spike ratio should become 1.0)
test_raw = raw.loc[[526]].copy()
test_raw["baseline_trades_per_day"] = 0
test_feats = build_features(test_raw)
test_scaled = prepare_for_model(test_feats, scaler)
print(test_feats["spike_ratio"].iloc[0])
print(explain_account(test_feats.iloc[0], test_scaled.iloc[0]))

Its trading is 28.0x its usual pace and it makes 880 trades per day.
1.0
It makes 880 trades per day and it is only 1 day old.


## Step 6: Score, flag the top 3%, and build the review table

Same functions and same 3% budget as Phase 5. An account flagged by both methods appears twice (once per method) with the same explanation.

In [69]:
scores_a = zscore_scores(scaled)                    # Method A
scores_b = isolation_forest_scores(model, scaled)   # Method B

flag_a = flag_top(scores_a)
flag_b = flag_top(scores_b)

print("Z-score flagged:", flag_a.sum())
print("Isolation Forest flagged:", flag_b.sum())
print("Flagged by both:", (flag_a & flag_b).sum())

Z-score flagged: 93
Isolation Forest flagged: 93
Flagged by both: 67


In [70]:
rows = []
for method, scores, flags in [("z-score", scores_a, flag_a),
                              ("isolation forest", scores_b, flag_b)]:
    for acc_id in scores[flags].index:
        rows.append({
            "account_id": acc_id,
            "suspicion_score": round(scores[acc_id], 3),
            "method": method,
            "explanation": explain_account(feats.loc[acc_id], scaled.loc[acc_id]),
        })

reviews = pd.DataFrame(rows).sort_values(["method", "suspicion_score"], ascending=[True, False])
print(reviews.shape)
reviews.groupby("method").head(3)

(186, 4)


,account_id,suspicion_score,method,explanation
159,ACC02495,0.747,isolation forest,Its trading is 23.9x its usual pace and it makes 317 trades per day.
174,ACC02831,0.725,isolation forest,Its trading is 29.8x its usual pace and it makes 187 trades per day.
155,ACC02370,0.720,isolation forest,Its trading is 25.6x its usual pace and it makes 139 trades per day.
72,ACC02593,13.605,z-score,It shares a device or IP with 12 other accounts.
73,ACC02629,13.605,z-score,It shares a device or IP with 12 other accounts.
84,ACC02831,12.871,z-score,Its trading is 29.8x its usual pace and it makes 187 trades per day.


In [71]:
print("Empty explanations:", (reviews["explanation"].str.len() == 0).sum())
reviews.to_csv("../data/flagged_case_reviews.csv", index=False)

Empty explanations: 0


## Step 7: Spot-check by hand

For each account below, check: (1) every number in the sentence matches the real values, (2) the wording reads naturally, (3) the two reasons look like the account's most suspicious features. Write a short note for each one.

In [72]:
pd.set_option("display.max_colwidth", None)

sample = pd.concat([
    reviews[reviews["method"] == "z-score"].sample(6, random_state=1),
    reviews[reviews["method"] == "isolation forest"].sample(6, random_state=1),
])

for _, r in sample.iterrows():
    print(r["account_id"], "|", r["method"], "|", r["explanation"])
    print("   ", feats.loc[r["account_id"]].round(2).to_dict())

ACC00215 | z-score | Its trade timing is unusually regular.
    {'account_age_days': 268.0, 'trades_per_day': 16.76, 'spike_ratio': 0.87, 'linked_account_count': 1.0, 'timing_regularity': 0.03}
ACC00958 | z-score | It makes 115 trades per day.
    {'account_age_days': 669.0, 'trades_per_day': 115.4, 'spike_ratio': 1.22, 'linked_account_count': 0.0, 'timing_regularity': 0.94}
ACC01511 | z-score | It shares a device or IP with 4 other accounts and it makes 24 trades per day.
    {'account_age_days': 430.0, 'trades_per_day': 24.43, 'spike_ratio': 1.23, 'linked_account_count': 4.0, 'timing_regularity': 1.27}
ACC01736 | z-score | Its trade timing is unusually regular.
    {'account_age_days': 264.0, 'trades_per_day': 8.09, 'spike_ratio': 1.11, 'linked_account_count': 0.0, 'timing_regularity': 0.01}
ACC02779 | z-score | It shares a device or IP with 3 other accounts and it makes 52 trades per day.
    {'account_age_days': 374.0, 'trades_per_day': 51.81, 'spike_ratio': 1.38, 'linked_account_c

### Wording and fallback counts

- Fallbacks: how often no feature was unusual enough.
- "only 0 days" should be 0 (we say "created today" instead).
- "only 1xx / 2xx days" shows whether a fairly old account is being called "only X days old". If many remain, we raise `MIN_SCORE` (for example to 1.5).

In [73]:
no_reason = reviews["explanation"].str.startswith("No feature")
print("Fallback explanations by method:")
print(no_reason.groupby(reviews["method"]).sum())

print("Say 'only 0 days':", reviews["explanation"].str.contains("only 0 days").sum())
print("Say 'created today':", reviews["explanation"].str.contains("created today").sum())
print("Say 'only 1xx or 2xx days':", reviews["explanation"].str.contains(r"only [12]\d\d days").sum())

Fallback explanations by method:
method
isolation forest    1
z-score             2
Name: explanation, dtype: int64
Say 'only 0 days': 0
Say 'created today': 2
Say 'only 1xx or 2xx days': 4


### Known limitation: z-score flags caused by a harmless direction

The z-score method uses the *absolute* z-score, so an account can be flagged for being unusual in a harmless direction (for example, very irregular timing). The explanation then describes other suspicious-looking features instead. We count how often that happens.

In [74]:
z_flagged = scaled[flag_a]
trigger = z_flagged.abs().idxmax(axis=1)   # the feature that caused each z-score flag

harmless = []
for acc_id in z_flagged.index:
    col = trigger[acc_id]
    if z_flagged.loc[acc_id, col] * direction[col] < 0:
        harmless.append(acc_id)

print("Z-score flags triggered by a harmless direction:", len(harmless), "of", len(z_flagged))
for acc_id in harmless:
    print(acc_id, "| trigger:", trigger[acc_id], "| value:", feats.loc[acc_id, trigger[acc_id]])
    print("   ", explain_account(feats.loc[acc_id], scaled.loc[acc_id]))

Z-score flags triggered by a harmless direction: 3 of 93
ACC00504 | trigger: timing_regularity | value: 1.8
    No feature looks clearly suspicious on its own; review the account's raw values.
ACC00530 | trigger: timing_regularity | value: 1.782
    No feature looks clearly suspicious on its own; review the account's raw values.
ACC01336 | trigger: timing_regularity | value: 2.0
    It makes 55 trades per day and it is only 7 days old.


## Step 8 (starts here): Check explanation quality against the labels

Labels are used **only now**, for checking, never to build explanations. First look at the exact label names so we can map each abuse type to the feature its explanation should mention.

In [75]:
labels = pd.read_csv("../data/labels.csv")
print(labels["anomaly_type"].value_counts())
print()
print(labels["difficulty"].value_counts())

anomaly_type
normal           3000
bot                33
multi_account      33
trial_abuse        33
Name: count, dtype: int64

difficulty
none       3000
obvious      51
subtle       48
Name: count, dtype: int64


In [76]:
labels = pd.read_csv("../data/labels.csv").set_index("account_id")

# words that show up in the sentence for each feature
mentions = {
    "account_age_days":     ["day old", "days old", "created today"],
    "trades_per_day":       ["trades per day"],
    "spike_ratio":          ["usual pace"],
    "linked_account_count": ["device or IP"],
    "timing_regularity":    ["timing"],
}

# which features each abuse type manipulates
manipulated = {
    "trial_abuse":   ["account_age_days", "trades_per_day", "spike_ratio"],
    "multi_account": ["linked_account_count"],
    "bot":           ["timing_regularity"],
}

def features_in(text):
    return [col for col, words in mentions.items() if any(w in text for w in words)]

# keep only flagged accounts that really are anomalies
check = reviews.join(labels, on="account_id")
check = check[check["anomaly_type"] != "normal"].copy()

check["hit"] = check.apply(
    lambda r: any(f in manipulated[r["anomaly_type"]] for f in features_in(r["explanation"])),
    axis=1)

print(check.groupby("method")["hit"].agg(["sum", "count", "mean"]))

                  sum  count  mean
method                            
isolation forest   66     66   1.0
z-score            84     84   1.0


In [77]:
print(check.groupby(["method", "anomaly_type"])["hit"].agg(["sum", "count", "mean"]))
print()
print(check.groupby(["method", "difficulty"])["hit"].agg(["sum", "count", "mean"]))

                                sum  count  mean
method           anomaly_type                   
isolation forest bot              5      5   1.0
                 multi_account   28     28   1.0
                 trial_abuse     33     33   1.0
z-score          bot             18     18   1.0
                 multi_account   33     33   1.0
                 trial_abuse     33     33   1.0

                             sum  count  mean
method           difficulty                  
isolation forest obvious      38     38   1.0
                 subtle       28     28   1.0
z-score          obvious      51     51   1.0
                 subtle       33     33   1.0


In [78]:
trial = check[check["anomaly_type"] == "trial_abuse"].copy()
trial["mentions_age"] = trial["explanation"].apply(lambda t: "account_age_days" in features_in(t))
print(trial.groupby("method")["mentions_age"].agg(["sum", "count"]))

                  sum  count
method                      
isolation forest    3     33
z-score             3     33


In [79]:
old = reviews[reviews["explanation"].str.contains(r"only [12]\d\d days")]
print(old[["account_id", "method", "explanation"]])

   account_id            method  \
99   ACC00201  isolation forest   
95   ACC00117  isolation forest   
4    ACC00201           z-score   
22   ACC00867           z-score   

                                                                    explanation  
99  It shares a device or IP with 8 other accounts and it is only 173 days old.  
95  It shares a device or IP with 2 other accounts and it is only 118 days old.  
4   It shares a device or IP with 8 other accounts and it is only 173 days old.  
22           Its trade timing is unusually regular and it is only 170 days old.  
